In [1]:
! pip install torch transformers

In [2]:
import numpy as np
import pandas as pd

In [3]:
import torch
from transformers import AutoTokenizer, AutoModel, AutoConfig


In [4]:
df=pd.read_csv('/kaggle/input/datasets/adithip2000/genre-generalised-and-tagged/final_processed_data.csv')

In [5]:
df.head()

,Unnamed: 0,genres,form,summary
0,0,"['sci-fi', 'drama', 'family']",book,"old major, the old boar on the manor farm, cal..."
1,1,"['sci-fi', 'drama']",book,"alex, a teenager living in near-future england..."
2,2,['drama'],book,the text of the plague is divided into five pa...
3,3,"['sci-fi', 'fantasy', 'drama']",book,the novel posits that space around the milky w...
4,4,"['sci-fi', 'fantasy', 'drama', 'family']",book,"ged is a young boy on gont, one of the larger ..."


In [6]:
# genre_map = {
#     # ---------------- DRAMA ----------------
#     "fiction": "drama",
#     "novel": "drama",
#     "drama": "drama",
#     "world cinema": "drama",

#     # ---------------- COMEDY ----------------
#     "comedy": "comedy",
#     "romantic comedy": "comedy",

#     # ---------------- ROMANCE ----------------
#     "romance novel": "romance",
#     "romance": "romance",
#     "romance film": "romance",
#     "romantic drama": "romance",

#     # ---------------- ACTION ----------------
#     "action": "action",
#     "action/adventure": "action",

#     # ---------------- ADVENTURE ----------------
#     "adventure": "adventure",
#     "adventure novel": "adventure",

#     # ---------------- THRILLER ----------------
#     "thriller": "thriller",
#     "suspense": "thriller",

#     # ---------------- HORROR ----------------
#     "horror": "horror",

#     # ---------------- SCI-FI ----------------
#     "science fiction": "sci-fi",
#     "speculative fiction": "sci-fi",
#     "dystopia": "sci-fi",
#     "alternate history": "sci-fi",

#     # ---------------- FANTASY ----------------
#     "fantasy": "fantasy",

#     # ---------------- CRIME ----------------
#     "crime fiction": "crime",

#     # ---------------- MYSTERY ----------------
#     "mystery": "mystery",
#     "detective fiction": "mystery",

#     # ---------------- HISTORY ----------------
#     "historical fiction": "history",
#     "historical novel": "history",

#     # ---------------- FAMILY ----------------
#     "children's literature": "family",
#     "family film": "family",
# }

In [7]:
df.shape

(49957, 4)

In [8]:
df.columns

Index(['Unnamed: 0', 'genres', 'form', 'summary'], dtype='object')

In [9]:
df.drop(['Unnamed: 0'],inplace=True,axis=1)

In [10]:
# from transformers import BertTokenizer as BToken
# tokenizer=BToken.from_pretrained('bert-base-uncased')
model_id = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)
#model = AutoModel.from_pretrained(model_id)
# Move the model to GPU if available
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#model.to(device)


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

In [11]:
type(df['genres'].iloc[0])

str

In [12]:
import ast
df['genres']=df['genres'].apply(ast.literal_eval)

In [13]:
type(df['genres'].iloc[0])

list

In [14]:
label_names=sorted(df['genres'].explode().unique())

In [15]:
print(label_names)

['action', 'adventure', 'comedy', 'crime', 'drama', 'family', 'fantasy', 'history', 'horror', 'mystery', 'romance', 'sci-fi', 'thriller']


In [16]:
from sklearn.preprocessing import MultiLabelBinarizer
mlb=MultiLabelBinarizer()
y=mlb.fit_transform(df['genres'])
print(y[0])


[0 0 0 0 1 1 0 0 0 0 0 1 0]


In [17]:
y.shape

(49957, 13)

In [18]:
y[12]

array([0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0])

In [19]:
from torch.utils.data import Dataset
class GenreDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, vectorizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.vectorizer = vectorizer
        self.max_len = max_len

        # 🔹 Use transform (NOT fit)
        bow_matrix = self.vectorizer.transform(self.texts)
        self.bow = torch.tensor(bow_matrix.toarray(), dtype=torch.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts.iloc[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            "text":text,
            "input_ids": encoding['input_ids'].squeeze(0),
            "attention_mask": encoding['attention_mask'].squeeze(0),
            "bow": self.bow[idx],
            "labels": torch.tensor(label, dtype=torch.float)
        }

In [20]:
label_counts=np.sum(y,axis=0)
for genre, count in zip(mlb.classes_, label_counts):
    print(genre, count)

action 6777
adventure 3599
comedy 10872
crime 5062
drama 27235
family 7090
fantasy 4426
history 1226
horror 4604
mystery 3614
romance 8919
sci-fi 7534
thriller 7925


In [21]:
total_samples=y.shape[0]
print(total_samples)


49957


In [22]:
X=df['summary']
# X=df['text']

In [23]:
from sklearn.model_selection import train_test_split as tts
X_train,X_test,Y_train,Y_test=tts(X,y,test_size=0.2,random_state=42)
X_val, X_test, Y_val, Y_test = tts(
    X_test,
    Y_test,
    test_size=0.5,
    random_state=42
)

In [24]:
print(X_train)

16256    in 2018 a prisoner named dante and his inmate ...
4204     the callum children spend their easter holiday...
6957     the enterprise is engaged in an unprecedented ...
9163     ike jerome, a 24-year-old architecture student...
14522    during a walk to the park raghuraman meets sud...
                               ...                        
11284    the novel continues on from the end of don't c...
44732    the film centers on themistocles and artemisia...
38158    cody weever grew up a child who loved films an...
860      the book's four main characters are ecological...
15795    the film begins with gayle, becky and judi per...
Name: summary, Length: 39965, dtype: object


In [25]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    max_features=8000,
    stop_words='english',
    min_df=5
)

vectorizer.fit(X_train)   # ONLY train

CountVectorizer(max_features=8000, min_df=5, stop_words='english')

In [26]:
train_dataset = GenreDataset(X_train, Y_train, tokenizer, vectorizer)
val_dataset   = GenreDataset(X_val, Y_val, tokenizer, vectorizer)
test_dataset  = GenreDataset(X_test, Y_test, tokenizer, vectorizer)

KeyboardInterrupt: 

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
vocab_size = len(vectorizer.get_feature_names_out())
bow_dim=train_dataset.bow.shape[1]
vocab=vectorizer.get_feature_names_out()
print(vocab[5:100])

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel

class topicGenreModel(nn.Module):
    def __init__(self, num_topics, num_genres, bow_dim, vocab_size, model_name="answerdotai/ModernBERT-base"):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        bert_dim = self.backbone.config.hidden_size
        self.bow_weights = nn.Parameter(torch.tensor(0.1))
        
        self.context_encoder = nn.Sequential(
            nn.Linear(bert_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        self.bow_encoder = nn.Sequential(
            nn.Linear(vocab_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # 🔹 Latent space (topics)
        self.fc_mu = nn.Linear(512, num_topics)
        self.fc_logvar = nn.Linear(512, num_topics)

        # 🔹 Topic → word matrix
        self.beta = nn.Parameter(torch.randn(num_topics, vocab_size))

        # 🔹 Topic → genre
        self.classifier = nn.Sequential(
            nn.Linear(num_topics + 256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_genres)
        )

    # ---------- Encode ----------
    def encode(self, bow, embedding):
        h_ctx = torch.relu(self.context_encoder(embedding))
        alpha = torch.sigmoid(self.bow_weights)
        
        if bow is not None:
            h_bow = torch.relu(self.bow_encoder(bow))
            if self.training:
                # Dropout-like mask for BoW features
                mask = (torch.rand(h_bow.size(0), 1).to(h_bow.device) > 0.3).float()
                h_bow = h_bow * mask
            h = torch.cat([h_ctx, alpha * h_bow], dim=1)
        else:
            h_bow = torch.zeros_like(h_ctx)
            h = torch.cat([h_ctx, h_bow], dim=1)
            
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)

        return mu, logvar, h_ctx

    # ---------- Reparameterization ----------
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    # ---------- Decode ----------
    def decode(self, topic_dist):
        logits = torch.matmul(topic_dist, self.beta)
        return torch.softmax(logits, dim=1)

    # ---------- Forward ----------
    def forward(self, input_ids, attention_mask, bow):
        # 🔹 BERT embedding (using CLS token)
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        embedding = outputs.last_hidden_state[:, 0, :]

        # 🔹 Encode
        mu, logvar, h_ctx = self.encode(bow, embedding)

        # 🔹 Sample topics
        z = self.reparameterize(mu, logvar)
        topic_dist = torch.softmax(z, dim=1)

        # 🔹 Genre prediction
        # Detaching topic distribution to prevent it from dominating the backbone gradients
        topic_dist_detached = topic_dist.detach()
        combined = torch.cat([h_ctx, 0.5 * topic_dist_detached], dim=1)
        genre_logits = self.classifier(combined)

        # 🔹 Word reconstruction
        word_dist = self.decode(topic_dist)

        return topic_dist, genre_logits, word_dist, mu, logvar

In [ ]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

   return {
            "input_ids": encoding['input_ids'].squeeze(0),
            "attention_mask": encoding['attention_mask'].squeeze(0),
            "bow": self.bow[idx],
            "labels": torch.tensor(label, dtype=torch.float)
        }

Remember, whole batch will be processed at once from the loader

In [ ]:
import torch
import numpy as np
import pandas as pd  # Optional: for pretty-printing results
from sklearn.metrics import f1_score, classification_report, recall_score

def validation(model, loader, device, num_labels, target_names=None, threshold=0.3):
    """
    Evaluates the model and returns overall and class-wise classification metrics.
    
    Args:
        target_names: List of strings (genre names). If provided, the report 
                      will use these names instead of "0", "1", "2"...
    """
    model.eval()
    
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            bow = batch['bow'].to(device)
            labels = batch['labels'].to(device).float()

            _, genre_logits, _, _, _ = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                bow=bow
            )

            probs = torch.sigmoid(genre_logits)
            all_probs.append(probs.detach().cpu())
            all_labels.append(labels.detach().cpu())
            
    all_probs = torch.cat(all_probs)
    all_labels = torch.cat(all_labels)
    
    final_preds = (all_probs > threshold).int().numpy()
    true_labels = all_labels.numpy()
    
    # --- Class-wise Metrics ---
    # Setting average=None returns an array of scores for each class
    class_sensitivity = recall_score(true_labels, final_preds, average=None, zero_division=0)
    class_f1 = f1_score(true_labels, final_preds, average=None, zero_division=0)
    
    # Detailed Dictionary Report
    # Setting output_dict=True allows you to access report['Genre_Name']['f1-score']
    full_report_dict = classification_report(
        true_labels, 
        final_preds, 
        target_names=target_names, 
        zero_division=0, 
        output_dict=True
    )

    # --- Summary Metrics ---
    metrics = {
        "f1_macro": f1_score(true_labels, final_preds, average="macro", zero_division=0),
        "sensitivity_macro": recall_score(true_labels, final_preds, average="macro", zero_division=0),
        "class_wise": {
            "sensitivity": class_sensitivity,
            "f1_score": class_f1
        },
        "full_report": full_report_dict,
        "all_probs": all_probs,
        "all_labels": all_labels
    }

    return metrics

In [ ]:
# vocab_size = len(vectorizer.get_feature_names_out())
# bow_dim=train_dataset.bow.shape[1]
num_genres=len(mlb.classes_)
print(f"Vocab size {vocab_size}")
print(f"bow dim {bow_dim}")
print(f"num_genres {num_genres}")
num_topics=32
print(f"num topics {num_topics}")
model=topicGenreModel(num_topics,num_genres,bow_dim,vocab_size)
state_dict=torch.load('/kaggle/input/models/adithip2000/final-ver/pytorch/default/1/checkpoint_v8_on_v7.pt',map_location=device)
model.load_state_dict(state_dict)
model.to(device)

    
    
    

In [ ]:

    # ---- VALIDATE ----
results= validation(
        model, test_loader, device, num_genres, threshold=0.5
    )
report_df = pd.DataFrame(results['full_report']).transpose()
print(report_df)
    # ---- PRINT ----
#print(f"Val Loss: {val_loss:.4f}")
#print(f"Val reconstruction loss: {avg_r:.4f}")
#print(f"Val KL divergence: {avg_kl:.4f}")
#print(f"Val classification loss: {avg_cl:.4f}")
   

In [ ]:
def inspect_sample(model, dataloader, vocab, label_names, device, threshold=0.2, idx=0, original_texts=None):
    
    model.eval()
    
    # Get one batch
    batch = next(iter(dataloader))
    
    # Select sample
    input_ids = batch['input_ids'][idx].unsqueeze(0).to(device)
    attention_mask = batch['attention_mask'][idx].unsqueeze(0).to(device)
    bow = batch['bow'][idx].unsqueeze(0).to(device)
    labels = batch['labels'][idx].to(device)
    text=batch['text'][idx]
    
    with torch.no_grad():
        topic_dist, genre_logits, word_dist, mu, logvar = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            bow=bow
        )
    
    # Predictions
    probs = torch.sigmoid(genre_logits)
    preds = (probs > threshold).int()
    
    # Convert to label names
    predicted_labels = [label_names[i] for i in range(len(label_names)) if preds[0][i] == 1]
    true_labels = [label_names[i] for i in range(len(label_names)) if labels[i] == 1]
    
    # Topic info
    topic_distribution = topic_dist[0].cpu().numpy()
    top_topic = torch.argmax(topic_dist, dim=1).item()
    
    # Top words
    def get_top_words(word_dist, vocab, top_k=10):
        top_indices = word_dist.squeeze().argsort(descending=True)[:top_k]
        return [vocab[i] for i in top_indices]
    
    top_words = get_top_words(word_dist, vocab)
    
    # Return values (useful if needed)
    return {
        "text":text,
        "true_labels": true_labels,
        "predicted_labels": predicted_labels,
        "topic_distribution": topic_distribution,
        "top_topic": top_topic,
        "top_words": top_words
    }

In [ ]:
for i in range(5,10):
    print("=================================================")
    print(inspect_sample(
    model=model,
    dataloader=test_loader,
    vocab=vocab,
    label_names=label_names,
    device=device,
    threshold=0.5,
    idx=i,
      # optional
))

for i, label in enumerate(label_names):
    print(f"{label}: f1={val_f1[i]:.3f}, precision={val_precision[i]:.3f}, recall={val_recall[i]:.3f}")
    # print(f"{label}: F1={val_f1[i]:.3f}, P={val_precision[i]:.3f}, R={val_recall[i]:.3f}")


def get_top_words(word_dist, vocab, top_k=10):
    top_indices = word_dist.argsort(descending=True)[:top_k]
    return [vocab[i] for i in top_indices]


for k in range(num_topics):
    top_words = get_top_words(topic_word_dist[k], vocab)
    print(f"Topic {k}: {top_words}")


Documents-> made of topics

Topics -> made of words

Yes — your intuition is basically correct, but let’s refine it so you don’t accidentally break training.


---

🧠 Core Principle (Important)

> In multi-objective models, the task with the largest loss dominates training.



So:

If reconstruction loss ≫ classification loss

Model will ignore classification



---

⚠️ But small correction to your idea

You said:

> “increase the contribution of the head with less loss”



👉 ❌ Not always correct
👉 ✔️ What you actually want is:

> Balance gradients / influence, not just raw loss values




---

🔥 In Your Case (Very Clear)

You have:

Reconstruction ≈ 8–9
Classification ≈ 0.3–0.5
KL ≈ 0.03

👉 So classification is too weak


---

✅ Correct Strategy for YOU

✔️ Make classification IMPORTANT

loss = (alpha * classi_loss) + recon_loss + (beta * kl_loss)

Start with:

alpha = 5.0
beta = 0.01


---

🔥 Why This Works

Now optimizer sees:

Classification gradients are stronger

Model cannot ignore genre prediction anymore



---

🔹 Better Mental Model

Instead of thinking:

> “increase smaller loss”



Think:

> “ensure each task contributes meaningfully to gradient updates”




---

🚀 Practical Ways to Balance Multi-Task Loss

✅ 1. Manual Weighting (what you're doing)

✔ Simple
✔ Works well

loss = α * L_class + L_recon + β * L_kl


---

✅ 2. Normalize Losses (VERY useful)

recon_loss = recon_loss / bow.sum()

👉 Prevents reconstruction from dominating


---

✅ 3. Monitor Each Loss (CRITICAL)

Always log:

print(f"Clf: {classi_loss:.3f}, Recon: {recon_loss:.3f}, KL: {kl_loss:.3f}")

👉 You want:

No loss ≈ 0

No loss dominating completely



---

🔥 4. Advanced (if you want later)

Dynamic weighting (learn weights automatically)

loss = w1 * L_class + w2 * L_recon

Where weights are learned

👉 Used in research (but not needed now)


---

🎯 For Your Project (Best Setup)

Final Loss:

loss = (5.0 * classi_loss) + recon_loss + (0.01 * kl_loss)


---

Validation:

Early stopping → F1 macro

Monitor:

val_loss

val_f1




---

Prediction:

preds = (sigmoid(logits) > 0.3)


---

🚨 Warning Signs to Watch

Symptom	Meaning

F1 = 0	classification ignored
Loss ↓ but F1 flat	wrong objective
Accuracy ↑ but F1 ↓	predicting all zeros



---

🧠 Final Insight (VERY IMPORTANT)

Your model is:

> “Using topics to support classification”



👉 So:

Classification MUST drive learning

Reconstruction is supporting signal, not main goal



---

🚀 Clean Way to Explain (Interview)

> “We used a weighted multi-objective loss where classification loss was upweighted to ensure the model prioritizes genre prediction, while reconstruction and KL divergence act as auxiliary regularizers.”




---

🔥 If you want next

I can help you:

Tune alpha automatically based on gradients

Or improve F1 further using threshold tuning + class weights (your next bottleneck)


# 🧠 PART 1: What is `dim=1`?

You’ve seen:

```python
torch.sum(..., dim=1)
torch.softmax(..., dim=1)
```

---

## 🎯 Think of tensors like tables

Example:

```text
word_dist shape = [batch_size, vocab_size]

= [32, 5000]
```

👉 Think:

```text
Row = one document  
Columns = words
```

---

## 🔹 dim=0 vs dim=1

| dim   | Meaning                            |
| ----- | ---------------------------------- |
| dim=0 | across rows (across batch)         |
| dim=1 | across columns (within one sample) |

---

## 🔥 Example

```python
x = [[1,2,3],
     [4,5,6]]
```

---

### sum(dim=1)

```text
[1+2+3, 4+5+6] = [6, 15]
```

👉 Row-wise (per document)

---

### sum(dim=0)

```text
[1+4, 2+5, 3+6] = [5, 7, 9]
```

👉 Column-wise

---

# 🎯 In your model

---

## 🔹 Softmax

```python
topic_dist = softmax(z, dim=1)
```

👉 For each document:

```text
sum of topics = 1
```

---

## 🔹 Reconstruction loss

```python
torch.sum(..., dim=1)
```

👉 Sum over all words **for each document**

---

## 🔥 Rule to remember

> `dim=1` = operate inside one sample

---

# 🧠 PART 2: Your classification weights (VERY GOOD 👍)

You did:

```python
class_weights = total_samples / (len(label_counts) * label_counts)
```

👉 This is **correct idea** (handling imbalance)

---

## But IMPORTANT correction

👉 You must compute weights using **TRAIN set only**

---

## ❌ You did:

```python
y = full dataset
```

---

## 🚨 Problem

* Data leakage
* Validation/test info used

---

## ✅ Correct

```python
label_counts = np.sum(Y_train, axis=0)
total_samples = Y_train.shape[0]

class_weights = total_samples / (len(label_counts) * label_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float)
```

---

# 🧠 PART 3: What is `pos_weight`?

You wrote:

```python
nn.BCEWithLogitsLoss(pos_weight=class_weights)
```

---

## 🎯 What it does

👉 Gives more importance to rare classes

---

## Example

If:

```text
Action = 1000 samples
Horror = 50 samples
```

👉 Model will ignore horror normally ❌

---

## With weights:

```text
Horror gets higher penalty if wrong
```

---

## 🔥 So:

```text
rare genres → more importance
```

---

# 🧠 Important detail

`pos_weight` applies to:

```text
only positive labels (1s)
```

---

# 🧠 PART 4: One small fix

Make sure:

```python
class_weights = class_weights.to(device)
```

---

# 🧠 FINAL SUMMARY

---

## 🔹 dim=1

👉 operate per sample (row-wise)

---

## 🔹 your class weights

✅ Good idea
❗ Use only training data

---

## 🔹 BCEWithLogitsLoss

👉 Handles multi-label + imbalance

---

# 💬 Final check

Answer this:

👉 Why do we use `dim=1` in softmax for `topic_dist`?

---

After that, I’ll help you:
👉 verify your loss is working correctly (very important next step) 🔥



# 🧠 FIRST: What are we trying to do?

You want:

```text
Document → Topics → Words
                   ↓
                Genre
```

So your model is learning:

* Topics = hidden concepts
* Words = what each topic contains
* Genre = based on topics

---

# 🧠 Key Variables (VERY IMPORTANT)

Let’s fix numbers:

```python
num_topics = 20
vocab_size = 5000
batch_size = 32
bert_dim = 768
```

---

# 🧠 What is `topic_dist`?

```text
topic_dist shape = [32, 20]
```

👉 For each document:

```text
[0.7, 0.2, 0.1, ...]
```

👉 Means:

* 70% Topic 1
* 20% Topic 2

---

# 🧠 What is `beta`?

```python
self.beta = nn.Parameter(torch.randn(num_topics, vocab_size))
```

---

## 🔥 This is VERY IMPORTANT

```text
beta shape = [20, 5000]
```

👉 Think:

```text
Topic 1 → [war, army, battle, ...]
Topic 2 → [love, heart, romance...]
```

👉 Each row = one topic
👉 Each column = one word

---

## ❓ What is `nn.Parameter`?

👉 It means:

> “This is a learnable weight — update it during training”

---

## Without it:

```python
self.beta = torch.randn(...)
```

❌ Model will NOT learn topics

---

## With it:

```python
nn.Parameter(...)
```

✅ Model learns topic-word relationships

---

# 🧠 Now the confusing part: `matmul`

---

## Code:

```python
logits = torch.matmul(topic_dist, self.beta)
```

---

## Dimensions:

```text
topic_dist = [32, 20]
beta       = [20, 5000]

result     = [32, 5000]
```

---

## What is happening?

👉 You are doing:

```text
Topics → Words
```

---

## Intuition

For ONE document:

```text
topic_dist = [0.7, 0.2, 0.1]
```

```text
beta:
Topic 1 → war, army
Topic 2 → love, heart
Topic 3 → crime, police
```

---

## Output:

```text
word_dist = combination of topics
```

👉 So:

```text
0.7 * Topic1 + 0.2 * Topic2 + 0.1 * Topic3
```

---

## 🔥 That’s what matmul does

👉 Weighted combination of topics → words

---

# 🧠 Why do we do this?

Because:

> A document = mixture of topics
> Topics = mixture of words

---

# 🧠 What is `nn.Sequential`?

---

## Your code:

```python
self.context_encoder = nn.Sequential(
    nn.Linear(bert_dim,256),
    nn.Softplus(),
    nn.Linear(256,256),
    nn.Softplus()
)
```

---

## What it means

Instead of writing:

```python
x = Linear(x)
x = activation(x)
x = Linear(x)
x = activation(x)
```

👉 `nn.Sequential` does it automatically

---

## Dimension flow

```text
[32, 768] → Linear → [32, 256]
           → Softplus
           → Linear → [32, 256]
           → Softplus
```

---

## Why do we need this?

👉 To convert:

```text
BERT embedding → features for topic learning
```

---

# 🧠 Same for BoW encoder

```python
self.bow_encoder = nn.Sequential(...)
```

---

## Input:

```text
bow = [32, 5000]
```

---

## Output:

```text
[32, 256]
```

---

# 🧠 Then we combine

```python
h = h_bow + h_ctx
```

👉 Both are:

```text
[32, 256]
```

---

# 🧠 Then latent space

```python
mu = [32, 20]
logvar = [32, 20]
```

👉 20 = number of topics

---

# 🧠 Final pipeline with dimensions

```text
Text → BERT → [32, 768]
                ↓
        context_encoder → [32, 256]

BoW → [32, 5000]
      ↓
bow_encoder → [32, 256]

Combine:
[32, 256] + [32, 256] → [32, 256]

↓
mu, logvar → [32, 20]

↓
z → [32, 20]

↓
softmax → topic_dist → [32, 20]

↓
matmul with beta

[32, 20] × [20, 5000] → [32, 5000]

↓
word distribution
```

---

# 🧠 Why vocab_size appears

👉 Because output is:

```text
probability of each word
```

---

# 🧠 Why num_topics appears

👉 Because:

```text
latent_dim = number of topics
```

---

# 🎯 Final intuition (MOST IMPORTANT)

---

## Model is learning:

👉 “Which topics explain this document?”

---

## Then:

👉 “Which words define those topics?”

---

## Then:

👉 “Which genre matches these topics?”

---


# 🧠 PART 1: Why VAE feels confusing (decoder, topics, etc.)

You said:

> “I only understood variation, not decoder or topic distribution”

👉 That’s the gap. Let’s fix it.

---

## 🎯 What you already know

VAE does:

```text
x → μ, σ → sample z
```

---

## ❗ What you’re missing

👉 What happens **after z**

---

## 🧠 FULL VAE picture (simple)

```text
Input → Encoder → μ, σ → z → Decoder → Output
```

---

## 🔥 Key idea

> Decoder tries to **recreate the input from z**

---

## 🧠 In normal VAE (images)

```text
z → Decoder → image
```

---

## 🧠 In YOUR case (topics)

```text
z → Decoder → words
```

---

## 🎯 So:

👉 `z` = topics
👉 decoder = converts topics → words

---

## 💡 That’s why you saw:

```python
topic_dist → beta → words
```

👉 That IS the decoder

---

# 🧠 What is “topic distribution”?

Very simple:

```text
z → softmax → topic_dist
```

---

## Example

```text
[0.7, 0.2, 0.1]
```

👉 Means:

* Topic 1 strong
* Topic 2 medium
* Topic 3 weak

---

## That’s it. No magic.

---

# 🧠 PART 2: Softplus vs ReLU (simple)

---

## 🔴 ReLU

```text
ReLU(x) = max(0, x)
```

---

### Problem

```text
x < 0 → output = 0
```

👉 Dead neurons
👉 No gradient

---

## 🟢 Softplus

```text
Softplus(x) = log(1 + exp(x))
```

---

### Behavior

* Always positive
* Smooth curve
* No sudden jump

---

# 🎯 Why Softplus is used here

👉 In VAE we deal with:

```text
probabilities, distributions
```

So we want:

| Property         | Needed? |
| ---------------- | ------- |
| Smooth gradients | ✅       |
| No zeros         | ✅       |
| Stable training  | ✅       |

---

## ❌ ReLU problem in VAE

* can output exact 0
* breaks variance / probability modeling

---

## ✅ Softplus advantage

* always > 0
* smooth
* stable

---

# 🧠 Simple analogy

* ReLU = ON/OFF switch
* Softplus = dimmer (smooth control)

---

# 🎯 Final takeaway

---

## About VAE

> Encoder finds hidden topics
> Decoder uses topics to recreate words

---

## About Softplus

> Used because VAE needs **smooth, positive values**

---






Perfect — this is the **best way to understand it**.
Let’s do a **tiny toy example** step-by-step. No abstractions.

---

# 🧠 Setup (very small world)

We define:

```text
vocab = [war, army, love, heart, crime]
vocab_size = 5
num_topics = 3
```

---

# 🧠 Step 1: One document

```text
Document: "war army war"
```

---

## BoW (counts)

```text
[war=2, army=1, love=0, heart=0, crime=0]
→ bow = [2, 1, 0, 0, 0]
```

---

## Normalize it

```text
bow_norm = [0.66, 0.33, 0, 0, 0]
```

👉 This is your **target**

---

# 🧠 Step 2: Model predicts topics

After encoder:

```text
z = [2.0, 0.5, -1.0]
```

---

## Apply softmax

```text
topic_dist = [0.7, 0.2, 0.1]
```

👉 Meaning:

* Topic 1 → strong
* Topic 2 → medium
* Topic 3 → weak

---

# 🧠 Step 3: Define topics (beta)

This is your:

```python
beta = nn.Parameter(...)
```

---

## Suppose model has learned:

```text
Topic 1 → [war=3, army=2, love=0, heart=0, crime=0]
Topic 2 → [war=0, army=0, love=3, heart=2, crime=0]
Topic 3 → [war=0, army=0, love=0, heart=0, crime=3]
```

---

# 🧠 Step 4: Decoder (THIS IS matmul)

We compute:

```text
word_logits = topic_dist × beta
```

---

## Calculation

```text
= 0.7 * Topic1
+ 0.2 * Topic2
+ 0.1 * Topic3
```

---

## Result

```text
word_logits ≈
[war=2.1, army=1.4, love=0.6, heart=0.4, crime=0.3]
```

---

## Apply softmax

```text
word_dist ≈
[0.5, 0.3, 0.1, 0.05, 0.05]
```

---

# 🧠 Step 5: Compare with real BoW

---

## Real (bow_norm)

```text
[0.66, 0.33, 0, 0, 0]
```

---

## Predicted (word_dist)

```text
[0.5, 0.3, 0.1, 0.05, 0.05]
```

---

# 🧠 Step 6: Loss

👉 Model sees:

```text
love, heart, crime should be LOWER
war, army should be HIGHER
```

---

👉 So it updates:

* topic_dist
* beta

---

# 🔥 What did we learn?

---

## 1. Topics are NOT predefined

👉 Model learns:

```text
Topic 1 = war-related
Topic 2 = romance
Topic 3 = crime
```

---

## 2. matmul meaning

👉 Mixing topics to produce words

```text
topics → words
```

---

## 3. Why BoW matters

👉 BoW tells model:

```text
“These words must be explained”
```

---

## 4. Why this creates topics

Because:

👉 Only way to reduce loss is:

```text
group similar words into same topic
```

---

# 🧠 Now your confusion points solved

---

## ❓ What is decoder?

👉 This:

```text
topic_dist × beta → word_dist
```

---

## ❓ What is topic_dist?

👉 Mixture of topics:

```text
[0.7, 0.2, 0.1]
```

---

## ❓ What is beta?

👉 Topic → word mapping

---

# 🎯 Final intuition (remember this)

👉 A document is:

```text
mixture of topics
```

👉 A topic is:

```text
mixture of words
```

---



## ✅ Step 1: Create classification loss (correct way)

Use **train labels only** to compute weights:

```python
import numpy as np
import torch
import torch.nn as nn

# from training data ONLY
label_counts = np.sum(Y_train, axis=0)
total_samples = Y_train.shape[0]

class_weights = total_samples / (len(label_counts) * label_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

classification_loss_fn = nn.BCEWithLogitsLoss(pos_weight=class_weights)
```

---

## 🧠 What this does

👉 If a genre is rare → model cares more about it
👉 Prevents ignoring small classes

---

# ✅ Step 2: Reconstruction loss (UNDERSTAND THIS WELL)

---

## Code

```python
def reconstruction_loss(word_dist, bow):
    # normalize bow → probabilities
    bow_norm = bow / (bow.sum(dim=1, keepdim=True) + 1e-10)

    loss = -torch.sum(
        bow_norm * torch.log(word_dist + 1e-10),
        dim=1
    ).mean()

    return loss
```

---

# 🧠 What is happening here (VERY IMPORTANT)

---

## 🔹 Inputs

| Variable    | Meaning                      |
| ----------- | ---------------------------- |
| `bow`       | real words (counts)          |
| `word_dist` | predicted word probabilities |

---

## 🔹 Example

Real document:

```text
bow = [war=2, army=1, love=0]
```

Normalize:

```text
bow_norm = [0.66, 0.33, 0]
```

---

Model predicts:

```text
word_dist = [0.5, 0.3, 0.1]
```

---

## 🔥 Loss says:

```text
“Make predicted distribution match real distribution”
```

---

# 🎯 Why this creates topics

Because:

👉 Only way to reduce this loss is:

```text
learn topics that generate correct words
```

---

# 🧠 Step 3: KL loss (small but needed)

```python
def kl_loss(mu, logvar):
    return -0.5 * torch.sum(
        1 + logvar - mu.pow(2) - logvar.exp(),
        dim=1
    ).mean()
```

---

## Why?

👉 Keeps topics:

* smooth
* not random
* not memorizing

---

# ✅ Step 4: FINAL LOSS (use this)

```python
def total_loss(word_dist, bow, genre_logits, labels, mu, logvar):

    recon = reconstruction_loss(word_dist, bow)

    kl = kl_loss(mu, logvar)

    cls = classification_loss_fn(genre_logits, labels)

    loss = cls + 0.1 * recon + 0.01 * kl

    return loss
```

---

# 🧠 Step 5: Use in training loop

```python
topic_dist, genre_logits, word_dist, mu, logvar = model(
    input_ids, attention_mask, bow
)

loss = total_loss(word_dist, bow, genre_logits, labels, mu, logvar)

loss.backward()
optimizer.step()
```

---

# 🧠 FINAL INTUITION (THIS IS EVERYTHING)

---

## 🔹 Classification loss

👉 “Topics must predict genre”

---

## 🔹 Reconstruction loss

👉 “Topics must explain words”

---

## 🔹 KL loss

👉 “Topics must be clean and structured”

---

# 🔥 Why your model will now work

Because now:

```text
BoW → forces topics to match words
Topics → used for genre
```

👉 So topics become:

* meaningful
* interpretable
* useful

---



# 🧠 Goal of reconstruction

We want:

```text
predicted words ≈ actual words
```

---

# 🧱 Step 1: Represent words as probabilities

Your model outputs:

```python
word_dist = softmax(...)
```

👉 So:

```text
word_dist = [0.5, 0.3, 0.2]
```

👉 This is:

> “Probability of each word”

---

Your BoW:

```text
[2, 1, 0]
```

Normalize:

```text
bow_norm = [0.66, 0.33, 0]
```

👉 This becomes:

> “True probability distribution”

---

# 🎯 So problem becomes:

> Compare two probability distributions

```text
True:      bow_norm
Predicted: word_dist
```

---

# 🧠 Step 2: How to compare distributions?

We use something called:

> **cross-entropy**

---

## Intuition of cross-entropy

For each word:

```text
if word is important → predicted prob should be high
```

---

## Formula (simple form)

```text
Loss = - Σ (true_prob × log(predicted_prob))
```

---

# 🧠 Step 3: Apply to your case

```text
Loss = - Σ bow_norm[i] × log(word_dist[i])
```

---

## Example

```text
bow_norm = [0.66, 0.33, 0]
word_dist = [0.5, 0.3, 0.2]
```

---

## Compute

```text
= -(0.66 * log(0.5) + 0.33 * log(0.3) + 0 * log(0.2))
```

👉 Notice:

* Only important words contribute
* irrelevant words ignored

---

# 🔥 Why log?

Because:

👉 We want to **punish wrong predictions strongly**

---

## Example

| prediction | log penalty   |
| ---------- | ------------- |
| 0.9        | small penalty |
| 0.1        | BIG penalty   |

---

👉 So:

> If model gives low probability to correct word → big loss

---

# 🧠 Step 4: Why negative sign?

Because:

```text
log(probability) is negative
```

👉 So we make loss positive:

```text
- log(...)
```

---

# 🧠 Step 5: Why sum over vocab?

Because:

👉 Document has multiple words

---

# 🧠 Step 6: Why `dim=1`?

Because:

```text
sum over words for each document
```

---

# 🧠 Step 7: Why mean?

Because:

```text
average loss across batch
```

---

# 🎯 Final formula becomes

```python
loss = -torch.sum(bow_norm * torch.log(word_dist), dim=1).mean()
```

---

# 🔥 Big intuition (IMPORTANT)

---

## What loss is saying

👉 For each word:

```text
“If this word appears a lot → give it high probability”
```

---

## What model learns

👉 To reduce loss, it must:

```text
topics → generate correct words
```

---

# 🧠 Why this creates topics

Because:

👉 Model cannot memorize each doc
👉 So it learns:

```text
Topic = group of co-occurring words
```

---

# 🎯 One-line summary

> Reconstruction loss = cross-entropy between real word distribution and predicted word distribution

---

# 💬 Final check

Answer this:

👉 If model predicts `word_dist = bow_norm` exactly, what will loss be?

---

If you get this, you’ve understood **the heart of topic modeling** 🔥


Here is the summary of the two architectures for your comparison. Both use a shared BERT encoder but handle the "Topic Bottleneck" differently.
Model A: The Linear Joint Model (Discriminative)
Focus: Simplicity and high classification accuracy.
Encoder: BERT (
) 
 Single Linear Layer 
 Softmax.
Bottleneck: A direct mapping from BERT to Topic Probabilities (
).
Head 1 (Genre): Linear layer mapping 
 Topics 
 
 Genres.
Head 2 (Words): Linear layer mapping 
 Topics 
 
 Vocabulary (Reconstruction).
Loss: 


.
Explainability: High. You see exactly which BERT features triggered a topic.
Model B: The CTM-based Joint Model (Generative/VAE)
Focus: High-quality, coherent topics and robust features.
Encoder: BERT (
) 
 Inference Net (2-3 Hidden Layers) 
 


 Layers.
Bottleneck: Reparameterization Trick. It samples from a distribution to create the Topic Vector (
).
Head 1 (Genre): Linear layer mapping 
 Topics 
 
 Genres.
Head 2 (Words): ProdLDA Decoder (Linear weight matrix 
) 
 
 Vocabulary.
Loss: 






.
Explainability: Very High. The KL-Divergence ensures topics are "clean" and not just noise that helps the classifier.
The Key Difference
Feature	Model A (Linear)	Model B (CTM/VAE)
Logic	"Find topics that match these labels."	"Find topics that explain the words and the labels."
Topic Quality	Can be "noisy" or repetitive.	Distinct and coherent word clusters.
Complexity	Easy to train; few hyperparameters.	Requires balancing the KL-Divergence weight.
Which implementation would you like to see the full code for first—the Linear or the CTM-based?
Something went wrong and an AI response wasn't generated.



Perfect 👍 — this is the **right way to learn regex** (line-by-line understanding).

We’ll go **one regex at a time** from the preprocessing pipeline we wrote for **BERT**.

I’ll explain like you are seeing regex for the first time 🙂

---

# ⭐ Regex 1 — Collapse repeated punctuation

```python
re.sub(r'([!?.]){2,}', r'\1', text)
```

## 🎯 Goal

Convert:

```
"Hello!!!!!" → "Hello!"
"What?????" → "What?"
```

---

## ✅ Break it down

### 🔹 `r''`

This means **raw string** in Python.

Why?

So Python does NOT treat `\` as escape.

Example:

```
"\n" → newline
r"\n" → backslash + n
```

---

### 🔹 `([!?.])`

This is a **group**.

Let’s break:

* `[!?.]` → character class
  Means **match ONE character that is either**

  * `!`
  * `?`
  * `.`

So it matches:

```
!
?
.
```

Then we put parentheses:

```
([!?.])
```

This means:

👉 "Capture this matched punctuation into Group-1"

---

### 🔹 `{2,}`

This is a **quantifier**

Means:

```
repeat previous pattern AT LEAST 2 times
```

So overall:

```
([!?.]){2,}
```

Means:

👉 Match same punctuation repeated **2 or more times**

Examples matched:

```
!!
???
.....
!!!!!
```

---

### 🔹 Replacement → `r'\1'`

This means:

👉 Replace whole match with **Group-1 value (single punctuation)**

Example:

```
Match → "!!!!!"
Group-1 → "!"
Replace → "!"
```

---

# ⭐ Regex 2 — Normalize elongated characters

```python
re.sub(r'(.)\1{2,}', r'\1\1', text)
```

## 🎯 Goal

Convert:

```
"goooooood" → "good"
"nooooo" → "noo"
```

---

## ✅ Break it down

### 🔹 `(.)`

* `.` → match **any character** (except newline)
* `( )` → capture group

So:

👉 capture ANY character into Group-1

Examples:

```
g
o
a
z
!
```

---

### 🔹 `\1`

This means:

👉 "same character as Group-1"

Very powerful concept.

Example:

If `(.)` matched `o`

Then `\1` means:

```
another 'o'
```

---

### 🔹 `{2,}`

Again:

```
repeat at least 2 times
```

So:

```
(.)\1{2,}
```

Means:

👉 one character + same character repeated **2 or more times**

Total = **3 or more repetitions**

Examples matched:

```
ooo
aaaa
!!!!!   (yes punctuation also)
```

---

### 🔹 Replacement → `\1\1`

Means:

👉 Replace long repetition with **exactly two characters**

Example:

```
"ooooooo" → "oo"
"aaaaa" → "aa"
```

Why keep 2?

Because:

* natural elongation signal preserved
* tokenizer behaves better

---

# ⭐ Regex 3 — Remove weird symbols

```python
re.sub(r'[^\w\s!?.:,;\'"-]', '', text)
```

This looks scary 😄 but simple.

---

## ✅ Break it down

### 🔹 `[ ... ]`

Character class → match **ONE character from set**

---

### 🔹 `^` (inside class)

Means **NEGATION**

So:

```
[^abc]
```

Means:

👉 match anything **except a or b or c**

---

### 🔹 `\w`

Means **word character**

Includes:

```
a-z
A-Z
0-9
_
```

---

### 🔹 `\s`

Means **whitespace**

Includes:

```
space
tab
newline
```

---

### 🔹 `!?.:,;\'"-`

These are punctuation we **allow**

---

## So whole regex means:

👉 Match any character that is **NOT**

* word character
* whitespace
* normal punctuation

So it removes:

```
@
#
$
%
~
*
emoji (depends)
```

Replacement is `''`

👉 means delete.

---

# ⭐ Regex 4 — Remove extra spaces

```python
re.sub(r'\s+', ' ', text)
```

## Break it down

* `\s` → whitespace
* `+` → one or more

So matches:

```
"   "
"\t\t"
"\n "
```

Replace with single space.

---

# 🔥 BIG CONCEPT YOU JUST LEARNED

Regex building blocks:

| Symbol        | Meaning          |
| ------------- | ---------------- |
| `.`           | any character    |
| `[]`          | character class  |
| `()`          | capture group    |
| `\1`          | backreference    |
| `{2,}`        | repeat 2 or more |
| `+`           | repeat 1 or more |
| `^` inside [] | NOT              |

---
